In [ ]:
%matplotlib widget

from pathlib import Path
import numpy as np
import flammkuchen as fl
import pandas as pd

from fimpylab import LightsheetExperiment

from matplotlib import  pyplot as plt
import seaborn as sns
from tqdm import tqdm
sns.set(style="ticks", palette="deep")
cols = sns.color_palette()
import ipywidgets as widgets

from lotr.utils import zscore
from lotr.pca import pca_and_phase, get_fictive_heading, fictive_heading_and_fit, \
        fit_phase_neurons,qap_sorting_and_phase
from circle_fit import hyper_fit
from lotr import LotrExperiment, A_FISH

In [ ]:
plt.close("all")
master =  Path(r"Z:\Hagar\e0075\v06")
master =  Path(r"Z:\Hagar\s11\e0075")
files = list(master.glob("*_f*"))
fish = files[2]
path = fish / 'suite2p' / '0007'

traces = fl.load(path / "filtered_traces.h5", "/detr")
#traces = fl.load(path / "selected.h5")

coords = fl.load(path / "data_from_suite2p_unfiltered.h5", "/coords")
anat = fl.load(path / "data_from_suite2p_unfiltered.h5", "/anatomy_stack")

exp = LotrExperiment(path)
fn = int(exp.fn)
beh_df = exp.behavior_log

t_start_s = 150
t_start_s = 50
t_lims = (t_start_s*exp.fn, exp.n_pts)
t_slice = slice(*t_lims)

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(anat.mean(0), vmax=1000, vmin=0)
plt.scatter(coords[:, 1], coords[:, 2], c=(0.9,)*3, s=1)
s1 = 50
s2 = 280
plt.axhline(s1)
plt.axhline(s2)

s3 = 40
s4 = 200
plt.axvline(s3)
plt.axvline(s4)

In [ ]:
sel_to_nan = (coords[:, 2] < s1) | (coords[:, 2] > s2) | (coords[:, 1] < s3) | (coords[:, 1] > s4)
traces[:, sel_to_nan] = 0

new_coords = coords[sel_to_nan]
plt.scatter(new_coords[:, 1], new_coords[:, 2], c=(0.5,)*3, s=1)


In [ ]:
cc_wnd = 4000
i_array = np.arange(t_slice.start, t_slice.stop, cc_wnd*fn)
cc_mats = np.zeros((traces.shape[1], traces.shape[1], len(i_array)))

for n, i in enumerate(i_array):
    cc_mats[:, :, n] = np.corrcoef(traces[i:i + cc_wnd*fn, :].T)
corr_mat = np.nanmean(cc_mats, 2)

selection_arr = np.zeros(traces.shape[1])

max_corr=-0.45

selection_arr[:] = np.nanmin(corr_mat, 0) < max_corr


plt.ylim(-0.15, 0.4)
plt.xlim(-0.3, 1.01)
plt.xlabel("cc. traces - motor regressor")
plt.ylabel("cc. d(traces)/dt - regressor")
sns.despine()

In [ ]:
selected = np.argwhere(selection_arr)[:, 0]
#selected = fl.load(path / "selected.h5")
print(len(selected))

In [ ]:
pcaed_t, phase_t, _, _ = pca_and_phase(traces[t_slice, selected].T, traces[t_slice, selected].T)
hf_c = hyper_fit(pcaed_t)
pcaed_t_all, _, _, _ = pca_and_phase(traces[t_slice, selected].T, traces[t_slice, :].T)


plt.figure(figsize=(7, 3))
thr = 35
sel = (pcaed_t[:, 0]**2+pcaed_t[:, 1]**2)**(1/2) > thr
plt.scatter(pcaed_t[:, 0], pcaed_t[:, 1], c=sel)
plt.scatter(pcaed_t_all[:, 0], pcaed_t_all[:, 1], edgecolor="k", facecolor="none", lw=0.2)
plt.axis("equal")

#selected = selected[sel]
# pcaed, phase = pca_and_phase(traces[t_slice, selected], traces[:, selected])
#pcaed_spont, phase_spont = pca_and_phase(traces[t_slice, selected], traces[t_slice, selected])
pcaed, phase, _, _ = pca_and_phase(traces[t_slice, selected], traces[:, selected])

x1 = hf_c[2]*np.cos(np.linspace(0, 2*np.pi, 100)) + hf_c[0]
x2 = hf_c[2]*np.sin(np.linspace(0, 2*np.pi, 100)) + hf_c[1]

plt.plot(x1, x2)

In [ ]:
new_selection_arr = (np.abs(np.sqrt((pcaed_t_all[:, 0] - hf_c[0])**2 + (pcaed_t_all[:, 1] - hf_c[1])**2) - hf_c[2]) < 10) | \
    ((np.sqrt((pcaed_t_all[:, 0] - hf_c[0])**2 + (pcaed_t_all[:, 1] - hf_c[1])**2) - hf_c[2]) > 0) | \
    (np.abs(pcaed_t_all[:, 1]) >20)
selected = np.argwhere(new_selection_arr)[:, 0]

In [ ]:
len(selected)

In [ ]:
pcaed, phase, _, _ = pca_and_phase(traces, traces)
mot_t_slice = slice(traces.shape[0] // 2, traces.shape[0])
f, axs = plt.subplots(1, 3, figsize=(9., 4.), sharex=True, sharey=True)

for i, s in enumerate([t_slice, mot_t_slice,  t_slice]):
    
    axs[i].plot(pcaed[s, 0], pcaed[s, 1], 
             c=(0.6,)*3, lw=0.5, zorder=-100) # , c=phase, cmap="twilight", lw=3)
    axs[i].scatter(pcaed[s, 0], pcaed[s, 1], 
                     c=phase[s], lw=0.5, s=5, cmap="twilight",) 
# plt.axis("equal")
sns.despine()

In [ ]:
%%time
import os
perm, com_phase = qap_sorting_and_phase(traces[:, selected], t_lims=t_lims)

phases_neuron, _ = fit_phase_neurons(traces[t_slice, selected], phase[t_slice])
perm_pca = np.argsort(phases_neuron)
os.system('say "Fit completed"')

l = 2
f, axs = plt.subplots(2,2, figsize=(7, 7), sharey=True)
# plt.subplot(121)
axs[0, 0].imshow(np.corrcoef(traces[t_slice, selected].T)[perm, :][:, perm], 
           vmax=1, vmin=-1, cmap="RdBu_r", aspect="auto")

axs[0, 1].imshow(traces[:, selected[perm]].T, cmap="gray_r", interpolation="none",
              aspect="auto", vmin=-l, vmax=l)

axs[1, 0].imshow(np.corrcoef(traces[t_slice, selected].T)[perm_pca, :][:, perm_pca], 
           vmax=1, vmin=-1, cmap="RdBu_r", aspect="auto")

axs[1,1].imshow(traces[:, selected[perm_pca]].T, cmap="gray_r", interpolation="none",
              aspect="auto", vmin=-l, vmax=l)


In [ ]:
unwrapped_phase = np.unwrap(phase)
unwrapped_com_phase = np.unwrap(com_phase)

In [ ]:
f, axs = plt.subplots(1, 2, figsize=(6, 3))
s = coords[:, 0] > 0
selection = np.full(coords.shape[0], False)
selection[selected] = True
all_phases = np.zeros(coords.shape[0])
all_phases[selected] = phases_neuron

all_perm = -np.ones(coords.shape[0])
all_perm[selected] = perm

axs[0].scatter(coords[s, 1], coords[s, 2], c=(0.5,)*3)
axs[0].scatter(coords[s, :][selection[s], 1], coords[s, :][selection[s], 2],
            c=all_phases[s][selection[s]], cmap="twilight")
axs[0].axis("equal")
axs[0].axis("off")

axs[1].scatter(coords[s, 1], coords[s, 2], c=(0.5,)*3)
axs[1].scatter(coords[s, :][selection[s], 1], coords[s, :][selection[s], 2],
            c=np.linspace(-np.pi, np.pi, sum(all_perm[s] >= 0)+1)[np.argsort(all_perm[s][all_perm[s] >= 0])] , 
               cmap="twilight")
axs[1].axis("equal")
axs[1].axis("off")

In [ ]:
from sklearn.decomposition import PCA
from circle_fit import hyper_fit
comp0, comp1 = 0, 1

traces_fit = traces[:, selected].T
traces_transform = traces_fit
if traces_transform is None:
    traces_transform = traces_fit

# Compute PCA and transform traces:
pca = PCA(n_components=5).fit(traces_fit)
pcaed_t = pca.transform(traces_transform)


# Fit circle:
hf_c = hyper_fit(pcaed[:, [comp0, comp1]])

# Compute phase, after subtracting center of the circle
phase_t = np.angle((pcaed_t[:, 0] - hf_c[0]) + 1j * (pcaed_t[:, 1] - hf_c[1]))

plt.figure(figsize=(7, 3))
plt.scatter(pcaed_t[:, 0], pcaed_t[:, 1], c=phase_t, cmap="twilight")
plt.axis("equal")

In [ ]:
fl.save(path / "selected.h5", selected)